### Imports

In [41]:
import numpy as np
import pandas as pd
import pymc3 as pm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from datetime import datetime
from datetime import timedelta

### Load files

In [43]:
insulin_data = pd.read_csv('processed-data/processed-tslim-bolus-data.csv').loc[:, ["CompletionDateTime", "ActualTotalBolusRequested", "CorrectionFactor", "TargetBG", "CarbSize", "InsulinToCarbRatio"]]

new_names = {
    "CompletionDateTime":"dateTime",
    "ActualTotalBolusRequested":"bolus",
    "CorrectionFactor":"ISF",
    "CarbSize":"guessedCarbohydrate",
    "InsulinToCarbRatio":"CIR"  
}
insulin_data = insulin_data.rename(columns = new_names)

# Round the datetime objects in the "timestamp" column to the nearest 5 minutes
interval = timedelta(minutes=5)
insulin_data["dateTime"] = insulin_data["dateTime"].astype("datetime64[ns]").apply(lambda x: x.round(interval))

# convert CIR to an integer 8, instead of e.g. 1:8
insulin_data["CIR"] = pd.to_numeric(insulin_data["CIR"].str.split(":").str[1], errors="coerce")

# Print the resulting dataframe
print(insulin_data.dtypes)
print(insulin_data.head)

dateTime               datetime64[ns]
bolus                         float64
ISF                             int64
TargetBG                        int64
guessedCarbohydrate             int64
CIR                             int64
dtype: object
<bound method NDFrame.head of                 dateTime  bolus  ISF  TargetBG  guessedCarbohydrate  CIR
0    2022-06-21 13:25:00   7.41   45       110                   60    8
1    2022-06-21 14:50:00   2.50   45       110                   20    8
2    2022-06-21 16:00:00   0.22   45       160                    0    0
3    2022-06-21 19:00:00   2.29   50       110                   25    8
4    2022-06-21 20:00:00   7.01   50       110                   65    8
...                  ...    ...  ...       ...                  ...  ...
1532 2022-12-21 20:55:00   1.83   52       110                   15    8
1533 2022-12-21 22:35:00   3.02    0         0                   34    0
1534 2022-12-22 07:40:00   0.57   50       110                    0    

In [27]:
blood_sugar_data = pd.read_csv('processed-data/processed-cgm-data.csv')
new_names = {
    "DateTime":"dateTime"
}
blood_sugar_data = blood_sugar_data.rename(columns = new_names)
blood_sugar_data["dateTime"] = blood_sugar_data["dateTime"].astype("datetime64[ns]")

# print(blood_sugar_data.dtypes)
# print(blood_sugar_data.head)

dateTime    datetime64[ns]
BG                 float64
dtype: object
<bound method NDFrame.head of                  dateTime     BG
0     2022-06-20 23:30:00  168.0
1     2022-06-20 23:35:00  153.0
2     2022-06-20 23:40:00  150.0
3     2022-06-20 23:45:00  142.0
4     2022-06-20 23:50:00  136.0
...                   ...    ...
51939 2022-12-22 13:00:00  154.0
51940 2022-12-22 13:05:00  147.0
51941 2022-12-22 13:10:00  141.0
51942 2022-12-22 13:15:00  135.0
51943 2022-12-22 13:20:00  133.0

[51944 rows x 2 columns]>


In [45]:
# first join the cgm and bolus data
combined_cgm_bolus_data = pd.merge(blood_sugar_data, insulin_data, on="dateTime", how="left")
print(combined_cgm_bolus_data.head)

<bound method NDFrame.head of                  dateTime     BG  bolus  ISF  TargetBG  guessedCarbohydrate  \
0     2022-06-20 23:30:00  168.0    NaN  NaN       NaN                  NaN   
1     2022-06-20 23:35:00  153.0    NaN  NaN       NaN                  NaN   
2     2022-06-20 23:40:00  150.0    NaN  NaN       NaN                  NaN   
3     2022-06-20 23:45:00  142.0    NaN  NaN       NaN                  NaN   
4     2022-06-20 23:50:00  136.0    NaN  NaN       NaN                  NaN   
...                   ...    ...    ...  ...       ...                  ...   
51976 2022-12-22 13:00:00  154.0    NaN  NaN       NaN                  NaN   
51977 2022-12-22 13:05:00  147.0    NaN  NaN       NaN                  NaN   
51978 2022-12-22 13:10:00  141.0    NaN  NaN       NaN                  NaN   
51979 2022-12-22 13:15:00  135.0    NaN  NaN       NaN                  NaN   
51980 2022-12-22 13:20:00  133.0    NaN  NaN       NaN                  NaN   

       CIR  
0      N

# Function that describes how insulin_data concentration decreases with time, assuming an insulin_data duration of 5 hours

In [ ]:
def insulin_levels(dose, time_interval=5, duration=5*60):
    """
    Calculate insulin_data levels at 5 minute increments over a specified duration after an insulin_data dose.
    
    Parameters:
    - dose (float): insulin_data dose in units
    - time_interval (int): Time interval in minutes (default: 5)
    - duration (int): Duration in minutes (default: 5*60)
    
    Returns:
    - insulin_levels (list): List of insulin_data levels at each time interval
    """
    # Initialize insulin_data levels list
    insulin_levels = [dose]
    
    # Calculate insulin_data levels at each time interval
    for i in range(1, int(duration / time_interval)):
        insulin_levels.append(insulin_levels[-1] * np.exp(-time_interval / 5))
    
    return insulin_levels

### Read in activity data

In [25]:
# activity_data = pd.read_csv('data/apple-health-data/ActivitySummary.csv')

# inthe form of step count, since this is a minute by minute summary as opposed to a daily summary
activity_data = pd.read_csv('data/apple-health-data/StepCount.csv').loc[:, ["startDate", "endDate", "value"]]

activity_data["startDate"] = activity_data["startDate"].astype("datetime64[ns]")
activity_data["endDate"] = activity_data["endDate"].astype("datetime64[ns]")

# time difference
# res = activity_data["endDate"].iloc[0] - activity_data["startDate"].iloc[0]
# print(res)



/var/folders/tw/05kybb3s7x59fvjzvrbfmt340000gn/T/ipykernel_88787/492998546.py:4: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  activity_data = pd.read_csv('data/apple-health-data/StepCount.csv').loc[:, ["startDate", "endDate", "value"]]


0 days 00:10:01


### Next

In [2]:

# Preprocess the data
X = np.column_stack((insulin_data, blood_sugar_data, activity_data))
X = (X - np.mean(X, axis=0)) / np.std(X, axis=0)
y = np.loadtxt('blood_glucose_data.txt')

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Define the model
with pm.Model() as model:
    # Prior distributions for model parameters
    alpha = pm.Normal('alpha', mu=0, sigma=10)
    beta = pm.Normal('beta', mu=0, sigma=10, shape=X_train.shape[1])
    
    # Likelihood function
    mu = alpha + pm.math.dot(X_train, beta)
    sigma = pm.HalfCauchy('sigma', beta=5)
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y_train)
    
    # Inference
    trace = pm.sample(draws=1000, tune=1000)
    
# Make predictions using the estimated model parameters
alpha_hat = np.mean(trace['alpha'])
beta_hat = np.mean(trace['beta'], axis=0)
sigma_hat = np.mean(trace['sigma'])

# Define a function to make predictions
def predict(X):
    return alpha_hat + np.dot(X, beta_hat)

# Make predictions on the test set
y_pred = predict(X_test)

# Evaluate the performance of the model
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Squared Error: {mse:.2f}')
print(f'Mean Absolute Error: {mae:.2f}')
print(f'R2 Score: {r2:.2f}')

<frozen importlib._bootstrap>:228: RuntimeWarning: numpy.ndarray size changed, may indicate binary incompatibility. Expected 16 from C header, got 96 from PyObject


FileNotFoundError: insulin_data.txt not found.